In [1]:
import pandas as pd
import numpy as np

input_df = pd.read_parquet("datasets/daily_total_sales.parquet")
input_df

,datetime,total_sales,day_of_week,month,is_weekend,lag_1,lag_7,rolling_mean_7
0,2017-01-09,67364.489004,0,1,0,83868.925011,92116.733997,77144.313428
1,2017-01-10,57878.826959,1,1,0,67364.489004,82144.183962,73677.833856
2,2017-01-11,67498.498082,2,1,0,57878.826959,83542.260046,71385.867861
3,2017-01-12,51648.812997,3,1,0,67498.498082,63309.811987,69720.010862
4,2017-01-13,68828.135029,4,1,0,51648.812997,75487.895984,68768.616440
...,...,...,...,...,...,...,...,...
168,2017-06-26,56422.055035,0,6,0,77040.755990,59910.121978,64084.486579
169,2017-06-27,51423.170007,1,6,0,56422.055035,60942.420006,62724.593722
170,2017-06-28,62261.489035,2,6,0,51423.170007,65345.439019,62284.029439
171,2017-06-29,48412.346025,3,6,0,62261.489035,49417.084986,62140.495302


In [2]:
input_df["target"] = input_df["total_sales"].shift(-1)
input_df = input_df.dropna().reset_index(drop=True)
input_df

,datetime,total_sales,day_of_week,month,is_weekend,lag_1,lag_7,rolling_mean_7,target
0,2017-01-09,67364.489004,0,1,0,83868.925011,92116.733997,77144.313428,57878.826959
1,2017-01-10,57878.826959,1,1,0,67364.489004,82144.183962,73677.833856,67498.498082
2,2017-01-11,67498.498082,2,1,0,57878.826959,83542.260046,71385.867861,51648.812997
3,2017-01-12,51648.812997,3,1,0,67498.498082,63309.811987,69720.010862,68828.135029
4,2017-01-13,68828.135029,4,1,0,51648.812997,75487.895984,68768.616440,81638.096985
...,...,...,...,...,...,...,...,...,...
167,2017-06-25,77040.755990,6,6,1,74768.635037,62433.863008,64582.781857,56422.055035
168,2017-06-26,56422.055035,0,6,0,77040.755990,59910.121978,64084.486579,51423.170007
169,2017-06-27,51423.170007,1,6,0,56422.055035,60942.420006,62724.593722,62261.489035
170,2017-06-28,62261.489035,2,6,0,51423.170007,65345.439019,62284.029439,48412.346025


In [13]:
feature_cols = input_df.columns.drop(["target", "datetime"])
feature_cols

Index(['total_sales', 'day_of_week', 'month', 'is_weekend', 'lag_1', 'lag_7',
       'rolling_mean_7'],
      dtype='object')

In [5]:
split_idx = int(len(input_df) * 0.8)
train_df = input_df.iloc[:split_idx]
test_df = input_df.iloc[split_idx:]

In [6]:
train_df.shape

(137, 9)

In [7]:
test_df.shape

(35, 9)

In [9]:
train_df.head()

,datetime,total_sales,day_of_week,month,is_weekend,lag_1,lag_7,rolling_mean_7,target
0,2017-01-09,67364.489004,0,1,0,83868.925011,92116.733997,77144.313428,57878.826959
1,2017-01-10,57878.826959,1,1,0,67364.489004,82144.183962,73677.833856,67498.498082
2,2017-01-11,67498.498082,2,1,0,57878.826959,83542.260046,71385.867861,51648.812997
3,2017-01-12,51648.812997,3,1,0,67498.498082,63309.811987,69720.010862,68828.135029
4,2017-01-13,68828.135029,4,1,0,51648.812997,75487.895984,68768.616440,81638.096985


In [10]:
test_df.head()

,datetime,total_sales,day_of_week,month,is_weekend,lag_1,lag_7,rolling_mean_7,target
137,2017-05-26,66857.272957,4,5,0,50866.386002,61768.318019,64245.694165,64975.386003
138,2017-05-27,64975.386003,5,5,1,66857.272957,77076.105035,62517.020017,66963.014047
139,2017-05-28,66963.014047,6,5,1,64975.386003,82892.873081,60241.325869,58001.266986
140,2017-05-29,58001.266986,0,5,0,66963.014047,59061.920045,60089.804004,53768.769046
141,2017-05-30,53768.769046,1,5,0,58001.266986,51037.604005,60479.970438,67443.445985


In [11]:
sorted_df = train_df.set_index("datetime").sort_index(ascending=True)
sorted_df.head()

,total_sales,day_of_week,month,is_weekend,lag_1,lag_7,rolling_mean_7,target
datetime,,,,,,,,
2017-01-09,67364.489004,0,1,0,83868.925011,92116.733997,77144.313428,57878.826959
2017-01-10,57878.826959,1,1,0,67364.489004,82144.183962,73677.833856,67498.498082
2017-01-11,67498.498082,2,1,0,57878.826959,83542.260046,71385.867861,51648.812997
2017-01-12,51648.812997,3,1,0,67498.498082,63309.811987,69720.010862,68828.135029
2017-01-13,68828.135029,4,1,0,51648.812997,75487.895984,68768.616440,81638.096985


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_error

X = sorted_df[feature_cols]
y = sorted_df["target"]

tscv = TimeSeriesSplit(n_splits=5, test_size=35)

cv_rmse = []
cv_mae = []

rf_model = RandomForestRegressor(n_estimators=100, random_state=42)

for fold, (train_idx, val_idx) in enumerate(tscv.split(X)):
    X_fold_tr, X_fold_val = X.iloc[train_idx], X.iloc[val_idx]
    y_fold_tr, y_fold_val = y.iloc[train_idx], y.iloc[val_idx]

    rf_model.fit(X_fold_tr, y_fold_tr)
    y_fold_pred = rf_model.predict(X_fold_val)

    fold_rmse = np.sqrt(mean_squared_error(y_fold_val, y_fold_pred))
    fold_mae = mean_absolute_error(y_fold_val, y_fold_pred)

    cv_rmse.append(fold_rmse)
    cv_mae.append(fold_mae)
    print(f"Fold {fold + 1} — RMSE: {fold_rmse:,.2f} | MAE: {fold_mae:,.2f}")

print(f"\nCV Mean RMSE : {np.mean(cv_rmse):,.2f} ± {np.std(cv_rmse):,.2f}")
print(f"CV Mean MAE  : {np.mean(cv_mae):,.2f} ± {np.std(cv_mae):,.2f}")


Fold 1 — RMSE: 10,819.61 | MAE: 7,642.64
Fold 2 — RMSE: 9,429.20 | MAE: 7,086.93
Fold 3 — RMSE: 8,706.57 | MAE: 5,409.15
Fold 4 — RMSE: 9,748.12 | MAE: 7,924.54
Fold 5 — RMSE: 7,234.88 | MAE: 5,585.22

CV Mean RMSE : 9,187.68 ± 1,190.00
CV Mean MAE  : 6,729.69 ± 1,043.31


In [15]:
test_df = test_df.set_index("datetime").sort_index(ascending=True)
test_df.head()

,total_sales,day_of_week,month,is_weekend,lag_1,lag_7,rolling_mean_7,target
datetime,,,,,,,,
2017-05-26,66857.272957,4,5,0,50866.386002,61768.318019,64245.694165,64975.386003
2017-05-27,64975.386003,5,5,1,66857.272957,77076.105035,62517.020017,66963.014047
2017-05-28,66963.014047,6,5,1,64975.386003,82892.873081,60241.325869,58001.266986
2017-05-29,58001.266986,0,5,0,66963.014047,59061.920045,60089.804004,53768.769046
2017-05-30,53768.769046,1,5,0,58001.266986,51037.604005,60479.970438,67443.445985


In [16]:
X_test = test_df[feature_cols]
y_test = test_df["target"]

In [17]:
rf_model.fit(X, y)

y_pred = rf_model.predict(X_test)

test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
test_mae  = mean_absolute_error(y_test, y_pred)

print(f"\nTest RMSE : {test_rmse:,.2f}")
print(f"Test MAE  : {test_mae:,.2f}")


Test RMSE : 6,482.02
Test MAE  : 4,874.68


In [18]:
importances = pd.Series(rf_model.feature_importances_, index=feature_cols)
print(f"\nFeature importances:\n{importances.sort_values(ascending=False)}")


Feature importances:
day_of_week       0.348739
lag_1             0.206716
total_sales       0.171036
lag_7             0.158147
rolling_mean_7    0.082627
month             0.025975
is_weekend        0.006761
dtype: float64


In [19]:
# Naive baseline — predict tomorrow = today
naive_rmse = np.sqrt(mean_squared_error(y_test, X_test["total_sales"]))
naive_mae  = mean_absolute_error(y_test, X_test["total_sales"])

print(f"Naive RMSE : {naive_rmse:,.2f}")
print(f"Naive MAE  : {naive_mae:,.2f}")

print(f"\nRF improvement over naive — RMSE: {((naive_rmse - test_rmse) / naive_rmse) * 100:.1f}%")
print(f"RF improvement over naive — MAE:  {((naive_mae - test_mae) / naive_mae) * 100:.1f}%")

Naive RMSE : 13,338.32
Naive MAE  : 11,574.26

RF improvement over naive — RMSE: 51.4%
RF improvement over naive — MAE:  57.9%
